In [ ]:
# ## workflow summary

# The full XGBoost time-series workflow is:

#  Prepare datetime
#  Create cyclical time features
#  Create lag features
#  Create rolling window features
#  Create interaction features
#  Split data by time
#  Run time-series cross-validation
#  Train final model
#  Predict on unseen test data
#  Evaluate model performance

In [ ]:
# # XGBoost Time-Series Forecasting Framework

# This notebook builds an XGBoost model for time-series forecasting.

# XGBoost does not understand time automatically, so we create:

# - cyclical time features
# - lag features
# - rolling window features
# - interaction features
# - time-series cross-validation

In [ ]:
# import libraries

import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
# ## 1. Prepare datetime column

# First, we convert the datetime column into proper datetime format and sort the data by time.

# This is very important because lag and rolling features depend on the correct time order.

In [ ]:
df["datetime"] = pd.to_datetime(df["datetime"])

df = df.sort_values("datetime").reset_index(drop=True)

target_col = "wholesale_price"   # change this to your target column

In [ ]:
# ## 2. Create cyclical time features

# Time is cyclical.

# For example:

# - hour 23 and hour 0 are close to each other
# - December and January are close to each other

# So instead of using only raw hour/month numbers, we use sine and cosine features.

In [ ]:
def create_time_features(df):
    df = df.copy()

    hour = df["datetime"].dt.hour
    dayofweek = df["datetime"].dt.dayofweek
    month = df["datetime"].dt.month
    dayofyear = df["datetime"].dt.dayofyear

    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    df["dow_sin"] = np.sin(2 * np.pi * dayofweek / 7)
    df["dow_cos"] = np.cos(2 * np.pi * dayofweek / 7)

    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)

    df["doy_sin"] = np.sin(2 * np.pi * dayofyear / 365)
    df["doy_cos"] = np.cos(2 * np.pi * dayofyear / 365)

    df["is_weekend"] = dayofweek.isin([5, 6]).astype(int)

    return df

In [ ]:
# ## 3. Create lag features

# Lag features help the model learn from past values.

# For example:

# - lag_1h = value from 1 hour before
# - lag_24h = value from same hour yesterday
# - lag_168h = value from same hour last week

# These are usually very powerful for electricity price forecasting.

In [ ]:
def create_lag_features(df, target_col):
    df = df.copy()

    df["lag_1h"] = df[target_col].shift(1)
    df["lag_2h"] = df[target_col].shift(2)
    df["lag_24h"] = df[target_col].shift(24)
    df["lag_48h"] = df[target_col].shift(48)
    df["lag_168h"] = df[target_col].shift(168)

    return df

In [ ]:
# ## 4. Create rolling window features

# Rolling features summarize recent behavior.

# They help the model understand:

# - recent average trend
# - recent volatility
# - recent minimum and maximum values

# We use `.shift(1)` before rolling to avoid data leakage.

In [ ]:
def create_rolling_features(df, target_col):
    df = df.copy()

    shifted_target = df[target_col].shift(1)

    df["rolling_mean_3h"] = shifted_target.rolling(3).mean()
    df["rolling_mean_6h"] = shifted_target.rolling(6).mean()
    df["rolling_mean_24h"] = shifted_target.rolling(24).mean()
    df["rolling_mean_168h"] = shifted_target.rolling(168).mean()

    df["rolling_std_24h"] = shifted_target.rolling(24).std()
    df["rolling_min_24h"] = shifted_target.rolling(24).min()
    df["rolling_max_24h"] = shifted_target.rolling(24).max()

    return df

In [ ]:
# ## 5. Create interaction features

# Interaction features combine two variables.

# They are useful when one feature affects the target differently depending on another feature.

# Example:

# - demand × temperature
# - wind generation × hour
# - demand × weekend

# Only create interactions that make real-world sense.

In [ ]:
def create_interaction_features(df):
    df = df.copy()

    if "demand" in df.columns and "temperature" in df.columns:
        df["demand_temp"] = df["demand"] * df["temperature"]

    if "wind_generation" in df.columns:
        df["wind_hour"] = df["wind_generation"] * df["hour_sin"]

    if "solar_generation" in df.columns:
        df["solar_hour"] = df["solar_generation"] * df["hour_cos"]

    if "demand" in df.columns:
        df["demand_weekend"] = df["demand"] * df["is_weekend"]

    if "hydro_storage" in df.columns and "demand" in df.columns:
        df["hydro_demand"] = df["hydro_storage"] * df["demand"]

    return df

In [ ]:
# ## 6. Apply all feature engineering steps

# Now we apply all feature engineering functions.

# After creating lag and rolling features, some rows will contain missing values because the first rows do not have enough past history.

# So we remove those rows using `dropna()`.

In [ ]:
df = create_time_features(df)
df = create_lag_features(df, target_col)
df = create_rolling_features(df, target_col)
df = create_interaction_features(df)

df = df.dropna().reset_index(drop=True)

df.head()

In [ ]:
# ## 7. Train-test split

# For time-series data, we should not use random train-test split.

# Instead, we split by date.

# Past data is used for training.
# Future data is used for testing.

In [ ]:
split_date = "2024-01-01"

train = df[df["datetime"] < split_date]
test = df[df["datetime"] >= split_date]

exclude_cols = ["datetime", target_col]

X_train = train.drop(columns=exclude_cols)
y_train = train[target_col]

X_test = test.drop(columns=exclude_cols)
y_test = test[target_col]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# ## 8. Time-series cross-validation

# Normal cross-validation randomly mixes data.

# That is not good for time-series because future information can leak into the past.

# So we use `TimeSeriesSplit`.

# Each fold trains on earlier data and validates on later data.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

cv_mae_scores = []
cv_rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):

    X_tr = X_train.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]

    X_val = X_train.iloc[val_idx]
    y_val = y_train.iloc[val_idx]

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )

    model.fit(X_tr, y_tr)

    val_pred = model.predict(X_val)

    mae = mean_absolute_error(y_val, val_pred)
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))

    cv_mae_scores.append(mae)
    cv_rmse_scores.append(rmse)

    print(f"Fold {fold + 1}")
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("-" * 30)

In [ ]:
# ## 9. Average cross-validation performance

# Now we calculate the average validation performance across all folds.

# This gives a more reliable estimate than one single train-validation split.

In [ ]:
print("Average CV MAE:", np.mean(cv_mae_scores))
print("Average CV RMSE:", np.mean(cv_rmse_scores))

In [ ]:
# ## 10. Train final XGBoost model

# After cross-validation, we train the final model using the full training data.

# The test data is still not used here.

In [ ]:
final_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

final_model.fit(X_train, y_train)

In [ ]:
# ## 11. Make predictions on test data

# Now we use the final trained model to predict future values in the test set.

In [ ]:
test = test.copy()

test["prediction"] = model.predict(X_test)

test[["datetime", target_col, "prediction"]].head()

In [ ]:
# ## 12. Evaluate model performance

# We evaluate the model using:

# - MAE: average absolute error
# - RMSE: penalizes large errors more strongly

# Lower values mean better performance.

In [ ]:
mae = mean_absolute_error(y_test, test["prediction"])
rmse = np.sqrt(mean_squared_error(y_test, test["prediction"]))

print("MAE:", mae)
print("RMSE:", rmse)